# Ablation (Table VII): CNN-only on FER+

ResNet-18 appearance branch only (no landmark graph). Reported in the paper: **69.20% accuracy, 68.00% weighted F1**.

Note: in the original run this number is the best checkpoint evaluated on the FER+ `validation` folder (8,341 images).
Run from the `notebooks/` folder with FER+ placed at `data/FER+`.


In [ ]:
import os
import cv2
import glob
import numpy as np
import pandas as pd
import mediapipe as mp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool
from torchvision import transforms
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import warnings
import json
from collections import Counter
warnings.filterwarnings("ignore")

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Configuration
DATA_DIR = "../data/FER+"
IMG_SIZE = 128
NUM_CLASSES = 8  # Classed of the dataset
BATCH_SIZE = 32  # Increased from 16
EPOCHS = 100  # Increased from 20
LR = 5e-4  # Reduced from 1e-3
WEIGHT_DECAY = 1e-4  # L2 regularization
K_NEIGHBORS = 10  # Increased from 8
EARLY_STOP_PATIENCE = 15

# Create directories for outputs
os.makedirs('../checkpoints', exist_ok=True)
os.makedirs('../outputs/cnn_only', exist_ok=True)

# Semantic Subset of 68 Landmarks
LANDMARK_SUBSET_INDICES = [
    # Jawline
    234, 93, 132, 58, 172, 136, 150, 149, 176, 148, 152, 377, 400, 378, 379, 365, 397, 288, 361, 323, 454,
    # Right Eyebrow
    70, 63, 105, 66, 107, 336, 296, 334, 293, 300,
    # Nose Bridge
    168, 6, 197, 195, 5,
    # Nose Bottom
    48, 115, 220, 45, 4, 275, 440, 344, 278,
    # Right Eye
    33, 246, 161, 160, 159, 158, 157, 173, 133, 155, 154, 153, 145, 144, 163, 7,
    # Left Eye
    362, 398, 384, 385, 386, 387, 388, 466, 263, 249, 390, 373, 374, 380, 381, 382,
    # Outer Lip
    0, 267, 269, 270, 409, 291, 375, 321, 405, 314, 17, 84, 181, 91, 146, 61, 185, 40, 39, 37
]
print(f"Selected {len(LANDMARK_SUBSET_INDICES)} landmarks for graph construction.")

# Reproducibility: fix random seeds (the original runs were not seeded, so expect small run-to-run variation)
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
class ImprovedFaceGraphDataset(Dataset):
    def __init__(self, root_dir, subset_indices=None, transform=None, k_neighbors=10, 
                 subset_fraction=1.0, augment_landmarks=False):
        self.root_dir = root_dir
        self.transform = transform
        self.subset_indices = subset_indices
        self.k_neighbors = k_neighbors
        self.augment_landmarks = augment_landmarks
        
        # Initialize MediaPipe
        self.mp_face_mesh = mp.solutions.face_mesh
        self.face_mesh = self.mp_face_mesh.FaceMesh(
            static_image_mode=True,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5
        )
        
        self.image_paths = []
        self.labels = []
        self.class_map = {}
        
        self._prepare_dataset(subset_fraction)
        
    def _prepare_dataset(self, subset_fraction):
        if not os.path.exists(self.root_dir):
            print(f"Path not found: {self.root_dir}")
            return
            
        classes = sorted([d for d in os.listdir(self.root_dir) 
                         if os.path.isdir(os.path.join(self.root_dir, d))])
        print(f"Classes found: {classes}")
        
        for idx, cls_name in enumerate(classes):
            self.class_map[cls_name] = idx
            cls_path = os.path.join(self.root_dir, cls_name)
            all_imgs = [f for f in os.listdir(cls_path) 
                       if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            
            if subset_fraction < 1.0:
                import random
                random.shuffle(all_imgs)
                num_to_take = int(len(all_imgs) * subset_fraction)
                all_imgs = all_imgs[:num_to_take]
                
            for img_name in all_imgs:
                self.image_paths.append(os.path.join(cls_path, img_name))
                self.labels.append(idx)
                
        print(f"Total images: {len(self.image_paths)}")
        
        # Print class distribution
        label_counts = Counter(self.labels)
        print("\nClass distribution:")
        for cls_name, cls_idx in sorted(self.class_map.items(), key=lambda x: x[1]):
            print(f"  {cls_name}: {label_counts[cls_idx]} samples")

    def _procrustes_normalize(self, landmarks):
        """Apply Procrustes alignment: center, scale, and optionally rotate landmarks"""
        # Center landmarks
        centroid = np.mean(landmarks, axis=0)
        centered = landmarks - centroid
        
        # Scale to unit variance
        scale = np.sqrt(np.sum(centered ** 2) / len(centered))
        if scale > 1e-6:
            normalized = centered / scale
        else:
            normalized = centered
            
        return normalized.astype(np.float32)
    
    def _augment_landmarks(self, landmarks):
        """Add Gaussian noise to landmarks for augmentation"""
        if self.augment_landmarks:
            noise = np.random.normal(0, 0.01, landmarks.shape)
            landmarks = landmarks + noise
        return landmarks

    def _get_landmarks(self, image):
        results = self.face_mesh.process(image)
        
        if not results.multi_face_landmarks:
            return None
        
        landmarks = results.multi_face_landmarks[0].landmark
        
        # Extract coordinates
        coords = []
        indices = self.subset_indices if self.subset_indices else range(len(landmarks))
        
        for idx in indices:
            if idx < len(landmarks):
                lm = landmarks[idx]
                coords.append([lm.x, lm.y, lm.z])
            
        coords = np.array(coords, dtype=np.float32)
        
        # Apply Procrustes normalization
        coords = self._procrustes_normalize(coords)
        
        # Apply augmentation if enabled
        coords = self._augment_landmarks(coords)
        
        return coords

    def _build_graph(self, landmark_coords):
        x = torch.tensor(landmark_coords, dtype=torch.float)
        
        # Dynamic k-NN edges with edge features
        try:
            nbrs = NearestNeighbors(n_neighbors=min(self.k_neighbors+1, len(landmark_coords)), 
                                   algorithm='ball_tree').fit(landmark_coords)
            distances, indices = nbrs.kneighbors(landmark_coords)
        except Exception as e:
            print(f"k-NN Error: {e}")
            return None
        
        edge_index = []
        edge_attr = []  # Store edge distances as features
        
        for i in range(len(landmark_coords)):
            for j_idx in range(1, len(indices[i])):
                j = indices[i, j_idx]
                dist = distances[i, j_idx]
                
                edge_index.append([i, j])
                edge_attr.append([dist])
                
                edge_index.append([j, i])
                edge_attr.append([dist])
        
        if not edge_index:
            edge_index = [[i, i] for i in range(len(landmark_coords))]
            edge_attr = [[0.0] for _ in range(len(landmark_coords))]
            
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)
        
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        image = cv2.imread(img_path)
        if image is None:
            return self.__getitem__((idx + 1) % len(self))
            
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        landmarks = self._get_landmarks(image_rgb)
        if landmarks is None:
            return self.__getitem__((idx + 1) % len(self))
            
        graph_data = self._build_graph(landmarks)
        if graph_data is None:
             return self.__getitem__((idx + 1) % len(self))

        # Image for CNN
        if self.transform:
            img_pil = Image.fromarray(image_rgb)
            img_tensor = self.transform(img_pil)
        else:
            img_resized = cv2.resize(image_rgb, (IMG_SIZE, IMG_SIZE))
            img_tensor = torch.tensor(img_resized).permute(2, 0, 1).float() / 255.0
            
        return img_tensor, graph_data, torch.tensor(label, dtype=torch.long)

In [ ]:
def collate_fn(batch):
    images, graphs, labels = zip(*batch)
    images = torch.stack(images)
    labels = torch.stack(labels)
    graphs = Batch.from_data_list(graphs)
    return images, graphs, labels

In [ ]:
class MultiHeadAttentionFusion(nn.Module):
    """Multi-head attention fusion with residual connections"""
    def __init__(self, img_dim, graph_dim, fusion_dim, num_heads=4):
        super(MultiHeadAttentionFusion, self).__init__()
        self.num_heads = num_heads
        self.fusion_dim = fusion_dim
        
        # Project to same dimension
        self.img_proj = nn.Linear(img_dim, fusion_dim)
        self.graph_proj = nn.Linear(graph_dim, fusion_dim)
        
        # Multi-head attention
        self.attention_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim * 2, fusion_dim // num_heads),
                nn.ReLU(),
                nn.Linear(fusion_dim // num_heads, 1)
            ) for _ in range(num_heads)
        ])
        
        # Learnable temperature
        self.temperature = nn.Parameter(torch.ones(1))
        
        # Residual fusion weights
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta = nn.Parameter(torch.tensor(0.5))
        self.gamma = nn.Parameter(torch.tensor(0.1))
        
        self.bn = nn.BatchNorm1d(fusion_dim)
        
    def forward(self, img_emb, graph_emb):
        # Project to same dimension
        img_h = F.relu(self.img_proj(img_emb))
        graph_h = F.relu(self.graph_proj(graph_emb))
        
        # Multi-head attention
        combined = torch.cat([img_h, graph_h], dim=1)
        
        attention_weights = []
        for head in self.attention_heads:
            weight = head(combined)
            attention_weights.append(weight)
        
        # Average attention across heads
        alpha = torch.sigmoid(torch.mean(torch.stack(attention_weights), dim=0) / self.temperature)
        
        # Gated fusion with residual
        z = self.alpha * alpha * img_h + self.beta * (1 - alpha) * graph_h + self.gamma * (img_h * graph_h)
        
        # Batch normalization
        z = self.bn(z)
        
        return z


class ImprovedFaceGNN(nn.Module):
    def __init__(self, num_classes):
        super(ImprovedFaceGNN, self).__init__()

        import torchvision.models as models
        
        # ✅ CNN backbone (ResNet-18)
        self.cnn = models.resnet18(pretrained=True)
        self.cnn.fc = nn.Identity()  # remove final FC layer

        # ✅ Classifier (only CNN features)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, graph=None):  # graph ignored
        cnn_feat = self.cnn(image)   # [batch, 512]
        out = self.classifier(cnn_feat)
        return out


# Print model summary
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel will have approximately {count_parameters(ImprovedFaceGNN(NUM_CLASSES)):,} trainable parameters")

In [ ]:
class LabelSmoothingCrossEntropy(nn.Module):
    """Cross entropy with label smoothing"""
    def __init__(self, epsilon=0.1, weight=None):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.epsilon = epsilon
        self.weight = weight
        
    def forward(self, outputs, targets):
        n_classes = outputs.size(-1)
        log_probs = F.log_softmax(outputs, dim=-1)
        
        # One-hot encoding with label smoothing
        targets_one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_smooth = (1 - self.epsilon) * targets_one_hot + self.epsilon / n_classes
        
        loss = -(targets_smooth * log_probs).sum(dim=-1)
        
        if self.weight is not None:
            loss = loss * self.weight[targets]
            
        return loss.mean()

In [ ]:
def train(model, loader, criterion, optimizer, scheduler, device, epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    # Step scheduler
    scheduler.step()
    
    return total_loss / len(loader), 100. * correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return total_loss / len(loader), 100. * correct / total, all_preds, all_labels

In [ ]:
# Enhanced Data Augmentation
transforms_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),

    transforms.ToTensor(),  # ✅ MUST COME BEFORE RandomErasing

    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),

    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transforms_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Initializing dataset with enhanced augmentation...")

from torchvision.datasets import ImageFolder


print("Initializing dataset (CNN-only)...")

train_dataset = ImageFolder(os.path.join(DATA_DIR, 'train'), transform=transforms_train)
val_dataset   = ImageFolder(os.path.join(DATA_DIR, 'validation'), transform=transforms_val)
test_dataset  = ImageFolder(os.path.join(DATA_DIR, 'test'), transform=transforms_val)

print(train_dataset.classes)

# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,  
    num_workers=0
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,  
    num_workers=0
)

print(f"\nTrain samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

# Compute class weights for handling imbalance
train_labels =  train_dataset.targets
class_counts = Counter(train_labels)
total_samples = len(train_labels)
class_weights = torch.tensor(
    [total_samples / (NUM_CLASSES * class_counts[i]) for i in range(NUM_CLASSES)],
    dtype=torch.float
).to(device)

print(f"\nClass weights for balanced training: {class_weights.cpu().numpy()}")

In [ ]:
# Model initialization
print("\nInitializing improved model...")
model = ImprovedFaceGNN(num_classes=NUM_CLASSES).to(device)
print(f"Total parameters: {count_parameters(model):,}")

# Loss function with label smoothing and class weights
criterion = LabelSmoothingCrossEntropy(epsilon=0.1, weight=class_weights)

# Optimizer with weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Cosine annealing scheduler with warm restarts
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# Training history
history = {
    'train_loss': [], 
    'val_loss': [], 
    'train_acc': [], 
    'val_acc': [],
    'learning_rates': []
}

# Early stopping
best_val_acc = 0.0
patience_counter = 0

print("\n" + "="*80)
print("Starting training with enhanced architecture and optimizations...")
print("="*80)

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, scheduler, device, epoch)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    
    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['learning_rates'].append(optimizer.param_groups[0]['lr'])
    
    print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:6.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:6.2f}% | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'history': history
        }, '../checkpoints/cnn_only_ferplus.pth')
        print(f"  → New best model saved! Val Acc: {val_acc:.2f}%")
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered after {epoch+1} epochs.")
        print(f"Best validation accuracy: {best_val_acc:.2f}%")
        break

print("\n" + "="*80)
print(f"Training completed! Best validation accuracy: {best_val_acc:.2f}%")
print("="*80)

# Save training history
with open('../outputs/cnn_only/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

In [ ]:
# Load best model for final evaluation
print("Loading best model for final evaluation...")
checkpoint = torch.load('../checkpoints/cnn_only_ferplus.pth')
model.load_state_dict(checkpoint['model_state_dict'])

# Final evaluation
val_loss, val_acc, all_preds, all_labels = evaluate(model, val_loader, criterion, device)

print(f"\nFinal Validation Results:")
print(f"  Loss: {val_loss:.4f}")
print(f"  Accuracy: {val_acc:.2f}%")

# Classification report
class_names = train_dataset.classes

print("\nPer-Class Performance:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss curves
axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# Accuracy curves
axes[0, 1].plot(history['train_acc'], label='Train Acc', linewidth=2)
axes[0, 1].plot(history['val_acc'], label='Val Acc', linewidth=2)
axes[0, 1].axhline(y=90, color='r', linestyle='--', label='Target (90%)', linewidth=1.5)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
axes[0, 1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# Learning rate schedule
axes[1, 0].plot(history['learning_rates'], linewidth=2, color='green')
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Learning Rate', fontsize=12)
axes[1, 0].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True, alpha=0.3)

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1, 1], cbar_kws={'label': 'Count'})
axes[1, 1].set_xlabel('Predicted', fontsize=12)
axes[1, 1].set_ylabel('True', fontsize=12)
axes[1, 1].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/cnn_only/training_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTraining visualizations saved to '../outputs/cnn_only/training_results.png'")

In [ ]:
# Visualize sample predictions
import random

model.eval()
num_samples = 20
sample_indices = random.sample(range(len(val_dataset)), min(num_samples, len(val_dataset)))

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()

with torch.no_grad():
    for idx, sample_idx in enumerate(sample_indices):
        img_tensor, label = val_dataset[sample_idx]
        
        # Prepare batch
        img_batch = img_tensor.unsqueeze(0).to(device)
        
        # Predict
        output = model(img_batch)
        probs = F.softmax(output, dim=1)
        confidence, predicted = probs.max(1)
        
        # Denormalize image for display
        img_display = img_tensor.permute(1, 2, 0).cpu().numpy()
        img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_display = np.clip(img_display, 0, 1)
        
        # Display
        axes[idx].imshow(img_display)
        axes[idx].axis('off')
        
        true_label = class_names[label]
        pred_label = class_names[predicted]
        conf = confidence.item() * 100
        
        color = 'green' if predicted == label else 'red'
        axes[idx].set_title(f"True: {true_label}\nPred: {pred_label} ({conf:.1f}%)",
                           fontsize=10, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/cnn_only/predictions_sample.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nSample predictions saved to '../outputs/cnn_only/predictions_sample.png'")

In [ ]:
# Final Summary
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)
print(f"\ CNN-only (ResNet-18)")
print(f"Total Parameters: {count_parameters(model):,}")
print(f"\nTraining Configuration:")
print(f"  - Epochs: {len(history['train_acc'])}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Initial Learning Rate: {LR}")
print(f"  - Weight Decay: {WEIGHT_DECAY}")
print(f"  - k-Neighbors: {K_NEIGHBORS}")
print(f"\nFinal Results:")
print(f"  - Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"  - Final Training Accuracy: {history['train_acc'][-1]:.2f}%")
print(f"  - Train-Val Gap: {abs(history['train_acc'][-1] - best_val_acc):.2f}%")

if best_val_acc >= 90:
    print(f"\n✅ SUCCESS: Target accuracy of 90% achieved!")
elif best_val_acc >= 85:
    print(f"\n⚠️  CLOSE: Achieved {best_val_acc:.2f}%, just below 90% target.")
    print(f"   Consider: longer training, larger batch size, or more data augmentation.")
else:
    print(f"\n❌ BELOW TARGET: Achieved {best_val_acc:.2f}%, target is 90%.")
    print(f"   Recommendations:")
    print(f"   - Increase model capacity (more layers/channels)")
    print(f"   - Add more aggressive data augmentation")
    print(f"   - Check for data quality issues")
    print(f"   - Consider using pre-trained CNN backbone")

print("\n" + "="*80)